# Rotating Square Patch (2D)

A square of fluid, initially in **rigid rotation** at `omega = 4 rad/s`, free
on all sides. Rigid rotation of a square is not an equilibrium: the corners
carry more speed than the edges, the free surface has to deform to balance the
pressure, and the patch grows four arms while its middle stays roughly square.
How long those arms stay coherent -- and whether the surface stays clean or
breaks into particle noise -- is the actual benchmark, and it is what separates
one scheme from another. `examples/incompressible/03-rotating-square-patch.ipynb`
is the same geometry run divergence-free (`--scheme divergenceFree`), which is
the comparison this case exists for.

Two things worth knowing before changing anything:

- **The box is three times the patch** (`L = 6` for a 2x2 square). The arms
  reach well past the patch's initial extent, and the domain is periodic, so a
  tighter box quietly turns this into a different experiment.
- **The corners are the test.** `--shape circle` is the null experiment: a
  circular patch in rigid rotation *is* an equilibrium, so nothing should
  happen. `--shape` takes any of the shapes listed below, so the sharper ones
  (`triangleIsosceles`, `star5`, `hexagram`) are the same benchmark turned up.

![](outputs/03-rotatingSquarePatch.gif)


## Every knob, and what it does

The parameters cell below is the whole command line of `03-rotating-square-patch.py` written out:
`CaseSpec` fields first, then `rotatingSquarePatchCase.params` -- the case's own physics knobs,
each of which is also a `--flag`. Anything not named there keeps the value in
`rotatingSquarePatchCase.defaults`/`.params`.

**Discretisation, time stepping and output** (`CaseSpec` fields, shared by every case)

| field | this notebook | what it does |
|---|---|---|
| `nx` | `192` | particles across the domain; the spacing is `dx = L / nx` |
| `dim` | `2` | this case is 2D |
| `L` | `6.0` | side of the (periodic) box |
| `n_h` | `4.0` | particles per support radius, i.e. how smooth the kernel is |
| `kernel` | `Wendland4` | SPH kernel |
| `integrationScheme` | `rungeKutta2` | time integrator |
| `scheme` | `deltaSPH` | the solver itself |
| `tLimit` | `1.0` | simulated end time; the loop runs `tLimit / dt` steps |
| `dt` | *set by the case* | left `None`: `initialConditions` picks it together with the sound speed |
| `adaptiveDt`, `cflFactor`, `minDt` | `True`, `0.3`, `1e-8` | CFL limiter around that `dt` |
| `plot`, `show`, `plotInterval` | `True`, `True`, `10` | render a frame every `plotInterval` steps |
| `store`, `storeMode`, `storeInterval` | `False`, `'states'`, `500` | HDF5 export; off here |

**The case's own parameters** (`--flag` on the script, `params=dict(...)` here)

| parameter | this notebook | what it does |
|---|---|---|
| `shape` | `'box'` | any key of `SHAPE_PRESETS`: `circle`, `box`, `roundedBox`, `rhombus`, `trapezoid`, `parallelogram`, `equilateralTriangle`, `triangleIsosceles`, `pentagon`, `hexagon`, `octogon`, `hexagram`, `star5`, `vesica`, `cutDisk`, `unevenCapsule`, `moon` |
| `size` | `1.0` | characteristic half-size of the shape |
| `aspectRatio` | `1.0` | squashes it in its second direction |
| `rotation` | `0.0` | degrees counter-clockwise, applied before anything else
| `omega` | `4.0` | angular velocity of the initial rigid rotation, rad/s, counter-clockwise |
| `rho0` | `1.0` | rest density |
| `targetDt` | `0.0005` | the timestep the run *asks* for; the sound speed is then chosen to make it the acoustic CFL limit |
| `inviscid`, `nu` | `True`, `0.0` | physical viscosity: `inviscid=True` leaves the scheme's own dissipation as the only one |
| `freeSurface` | `True` | surface detection, on for a case with a free surface |
| `band` | `0` | particle layers of boundary padding around the domain |
| `markerSize` | `8` | plot only: particle marker size |


**Three things this family does differently from the compressible notebooks**
(they will bite if `../compressible/08-Hydrostatic.ipynb` is copied unread):

1. The IC cell has a **fourth call**, `rotatingSquarePatchCase.initialConditions(ctx, system)`.
   That is where `setupWeaklyCompressibleTimestep` picks the sound speed and
   `config.dt` *together* from `targetDt` -- weakly compressible SPH is free to
   choose its own stiffness, so the timestep is fixed first and `c0` follows
   from the acoustic CFL. Skip it and `config.dt` stays `None`, and the patch is never given its rotation, since `initialConditions` is also where the rigid-rotation velocity field is stamped on.
2. **The loop is `range(nSteps)`.** No case in this family has a `timestep`
   hook, so `dt` is fixed for the whole run after step 1 and `while t < tLimit`
   would be the wrong shape.
3. Plotting calls `buildFieldPlotter`/`refreshFieldPlotter` on `VELOCITY_DENSITY_FIELDS`
   directly rather than `rotatingSquarePatchCase.setupPlot`/`updatePlot`, which go through
   `openWindow`/`pumpEvents` and do not live-update inside a Jupyter cell in
   this environment -- `08-Hydrostatic.ipynb` explains that in full.

Precision note: switching between single and double precision is controlled in
the import cell below. Because precision is set when core modules/kernels are
initialized, any precision change requires a kernel restart.

In [ ]:
%matplotlib inline
from warpSPHBootstrap import bootstrap
rt = bootstrap(precision='float32', verbose=True)

from warpSPH import *
from warpSPH.cases.rotatingSquarePatch import rotatingSquarePatchCase
from warpSPH.cases.plotting import buildFieldPlotter, refreshFieldPlotter
from warpSPH.runner import CaseSpec, buildContext, encodeFrames
from warpSPH.io import createOutFile, prepExport, writeInitialData, writeFrame
from warpSPH.cases.weaklyCompressible import VELOCITY_DENSITY_FIELDS

import os
import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.autonotebook import tqdm

In [ ]:
# Every knob you'd otherwise reach for as a `--flag` on
# `03-rotating-square-patch.py`, made explicit and editable here -- the table in the
# intro cell says what each one does. `rotatingSquarePatchCase.defaults`/`.params` are
# the same values the CLI script starts from.
spec = CaseSpec(caseName=rotatingSquarePatchCase.name, scheme=rotatingSquarePatchCase.scheme,
                params=dict(rotatingSquarePatchCase.params)) \
    .merged(**rotatingSquarePatchCase.defaults)

spec = spec.merged(
    # --- discretisation ------------------------------------------------
    nx=192,
    dim=2,
    # 3x the patch: the arms grow well past its initial extent, and the
    # domain is periodic.
    L=6.0,

    # --- time stepping ---------------------------------------------------
    tLimit=1.0,

    # --- output --------------------------------------------------------------
    caseName='03-rotatingSquarePatch',
    plot=True, show=True, plotInterval=10,
    store=False,

    # --- the patch's own knobs ------------------------------------------------
    params=dict(
        # the patch: `box` at aspect 1 is the 2x2 square of the benchmark
        shape='box', size=1.0, aspectRatio=1.0, rotation=0.0,
        # how fast it spins
        omega=4.0,
        # the fluid
        freeSurface=True, rho0=1.0, targetDt=0.0005, inviscid=True, nu=0.0,
        markerSize=8,
    ),
)
spec

In [ ]:
# Initial-condition generation: explicit, using the real case code
# (`rotatingSquarePatchCase.buildSystem`), not re-derived here.
#
# `initialConditions` is the call the compressible notebooks do not have:
# the patch is given its rigid-rotation velocity field there, and
# it is where the sound speed and `config.dt` are chosen together from
# `targetDt`, so skipping it leaves `config.dt` unset.
ctx = buildContext(rotatingSquarePatchCase, spec)
rotatingSquarePatchCase.configureScheme(ctx)
system = rotatingSquarePatchCase.buildSystem(ctx)
rotatingSquarePatchCase.initialConditions(ctx, system)
runningState = system.initializeNewState()

print(f'dt = {float(ctx.config.dt):.3e}, '
      f'c0 = {ctx.schemeConfig.fluid.fixedSoundSpeed:.3f}, '
      f'{len(runningState.state.positions)} particles')

In [ ]:
# What was actually built: the sampled regions, fluid and boundary, against the
# domain (black) the run is periodic in. This is the cell to look at when a
# geometry parameter above did something other than what it sounded like.
figure, axis = plt.subplots(1, 1, figsize=(5, 5), squeeze=False)
plotRegions(ctx.scratch['regions'], axis[0, 0], plotFluid=True, plotParticles=True)
domain = ctx.config.domain
axis[0, 0].set_aspect('equal')
axis[0, 0].set_xlim(domain.min[0].item(), domain.max[0].item())
axis[0, 0].set_ylim(domain.min[1].item(), domain.max[1].item())
axis[0, 0].set_title(f'{len(ctx.scratch["regions"])} regions, '
                     f'{len(runningState.state.positions)} particles')
figure.tight_layout()

In [ ]:
# Export/plot setup via the same generic hooks `warpSPH.runner.run()` uses
# internally -- nothing here is re-derived, only called explicitly.
ctx.exportPath = prepExport(spec.caseName, ctx.config, ctx.schemeConfig, ctx.scheme, ctx.exportFunction)
spec.save(os.path.join(ctx.exportPath, 'caseSpec.json'))
print(f'exporting to {ctx.exportPath}')

# Direct buildFieldPlotter(VELOCITY_DENSITY_FIELDS), not rotatingSquarePatchCase.setupPlot -- see the intro cell
# for why.
plotter = None
if spec.plot:
    ctx.imagePath = os.path.join(ctx.exportPath, 'images')
    os.makedirs(ctx.imagePath, exist_ok=True)
    plotter = buildFieldPlotter(ctx, runningState, VELOCITY_DENSITY_FIELDS)

outFile = None
groups = None
if spec.store and spec.storeMode == 'trajectory':
    extraData = rotatingSquarePatchCase.extraData(ctx, runningState)
    outFile = createOutFile(ctx.exportPath)
    groups = writeInitialData(ctx.exportPath, outFile, ctx.scheme, ctx.config, ctx.schemeConfig,
                              spec, runningState, extraData=extraData, extraFields=rotatingSquarePatchCase.extraFields)

In [ ]:
# The step loop, visible and editable. This is the same call
# `warpSPH.runner.runner._run` makes internally, unrolled here so a
# perturbation or an extra diagnostic can be injected directly around it.
dt = ctx.config.dt if isinstance(ctx.config.dt, float) else ctx.config.dt.cpu().item()
nSteps = int(spec.tLimit / dt)
storeSteps = max(1, int(spec.exportInterval / dt)) if spec.storeMode == 'trajectory' \
    else max(1, spec.storeInterval)

trajectory = []
angularMomentum = []
for i in (tq := tqdm(range(nSteps), leave=True)):
    # <-- hook point ---------------------------------------------------------
    stepResult = ctx.integrator.function(
        state=runningState, f=ctx.stepFunction, dt=ctx.config.dt,
        config=ctx.config, schemeConfig=ctx.schemeConfig, verbose=False,
    )
    runningState = stepResult.state
    # -------------------------------------------------------------------------

    # Injected at the hook point: total angular momentum. Rigid rotation of an
    # isolated patch conserves it exactly, so the drift below is a direct
    # measure of what the scheme's dissipation and the free-surface treatment
    # are costing -- and it is not in `diagnostics`, which is the point of
    # having the loop unrolled.
    particles = runningState.state
    angularMomentum.append(float((particles.masses
                                  * (particles.positions[:, 0] * particles.velocities[:, 1]
                                     - particles.positions[:, 1] * particles.velocities[:, 0])
                                  ).sum()))
    tScalar = runningState.t.item() if torch.is_tensor(runningState.t) else runningState.t
    row = rotatingSquarePatchCase.diagnostics(ctx, runningState)
    trajectory.append(dict(row, step=i, t=tScalar))
    tq.set_description(f"t: {tScalar:.4f}, " + ", ".join(f"{k}: {v:.4f}" for k, v in row.items()))

    if plotter is not None and (i % spec.plotInterval == 0 or i == nSteps - 1):
        refreshFieldPlotter(ctx, runningState, plotter, VELOCITY_DENSITY_FIELDS, step=i)

    if outFile is not None and (i % storeSteps == 0 or i == nSteps - 1):
        writeFrame(groups, i, stepResult.state, stepResult.stages, config=ctx.config,
                  schemeConfig=ctx.schemeConfig, uniqueParticles=True, writeStages=False,
                  extraFields=rotatingSquarePatchCase.extraFields)

In [ ]:
if outFile is not None:
    outFile.close()

if spec.plot:
    encodeFrames(ctx.imagePath, ctx.exportPath)

## What the run should show

The patch grows four arms, one per corner, which thin and curl in the direction
of rotation. Two numbers say whether that happened for the right reasons:

In [ ]:
# Angular momentum against its initial value (left) and the usual density
# bounds (right). L/L(0) drifting downwards is dissipation; a jump is the free
# surface losing particles off the arms.
figure, axis = plt.subplots(1, 2, figsize=(11, 3.5))
t = [row['t'] for row in trajectory]
axis[0].plot(t, np.array(angularMomentum) / angularMomentum[0])
axis[0].axhline(1.0, color='black', ls=':')
axis[0].set_xlabel('t'); axis[0].set_ylabel(r'$L(t) / L(0)$')
axis[1].plot(t, [row['maxDensity'] for row in trajectory], label='max')
axis[1].plot(t, [row['minDensity'] for row in trajectory], label='min')
axis[1].axhspan(0.99, 1.01, color='green', alpha=0.1, label=r'$\pm 1\%$')
axis[1].set_xlabel('t'); axis[1].set_ylabel(r'$\rho$'); axis[1].legend()
figure.tight_layout()